In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path('code').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ecg import (
    RECORDS_FOLDER,
    Subject,
    WindowManager,
    align_peaks,
    detect_qrs_for_subjects,
    extract_average_energy,
    get_available_patients,
    initialize_templates,
    load_all_subjects,
    load_patient_table,
    plot_peak_markers,
    plot_pt_steps,
    read_subject_data,
    synchronize_peaks,
    update_templates,
)
from ecg.features import split_annotations_by_type
from ecg.plotting import plot_age_distribution, plot_annotation_counts, plot_gender_distribution

## MIT-BIH Arrhythmia Database

### Arrhythmia Database contains 48 half-hour excerpts of two-channel ambulatory ECG recordings


### Informações dos pacientes


In [ ]:
df = load_patient_table(RECORDS_FOLDER)
df.head()

### Leitura dos dados


##### Data sample


In [ ]:
available_patients = get_available_patients(RECORDS_FOLDER)

subject_id = available_patients[2]
signal, fs, annotation = read_subject_data(subject_id, verbose=True)

signal_channel_1 = signal[:, 0]
signal_channel_2 = signal[:, 1]

In [ ]:
print(annotation.sample[:10])
print(annotation.symbol[:10])
print(set(annotation.symbol))

###### Getting all subjects data


In [ ]:
subjects: list[Subject] = load_all_subjects(RECORDS_FOLDER)
len(subjects)

#### Adaptive Threshold


In [ ]:
detected_indices = detect_qrs_for_subjects(subjects)
print(len(detected_indices))
print(len(detected_indices[0]))

In [ ]:
target_subject = subjects[0]
target_indices = detected_indices[0]
plot_peak_markers(target_subject.integrated_signal[:1000, 0], target_indices[:3])

##### Post-Processing


In [ ]:
real_peaks = align_peaks(target_indices, target_subject.raw_signal[:, 0], search_window_ms=25, fs=target_subject.fs)
print(target_subject.annotations.sample[1:], target_subject.annotations.sample[1:].shape)
print(real_peaks, real_peaks.shape)

In [ ]:
plot_peak_markers(target_subject.raw_signal[:1000, 0], real_peaks[:3])

Alguns picos naturalmente não vão ser detectados pelo algoritmo de Pan-Tompkins. Por isso, é necessário fazer a distinção de quais picos foram realmente detectados e quais não foram.

Para isso colocamos um período de 150ms entre um pico real e um pico detectado. Se ele não estiver neste intervalo podemos considerá-lo um falso positivo.


In [ ]:
synced_r_peaks_indices, synced_integrated_peaks_indices, synced_labels = synchronize_peaks(
    annotated_peaks=target_subject.annotations.sample,
    labels=target_subject.annotations.symbol,
    detected_r_peaks=real_peaks,
    detected_int_peaks=target_indices,
    fs=target_subject.fs,
)
len(synced_r_peaks_indices), len(synced_labels)

Windowing


In [ ]:
window_manager = WindowManager(
    synced_filtered_peaks=synced_r_peaks_indices,
    synced_labels=synced_labels,
    synced_integrated_peaks=synced_integrated_peaks_indices,
    integrated_signal=target_subject.integrated_signal[:, 0],
    filtered_signal=target_subject.filtered_signal[:, 0],
    window_span_ms=400,
    fs=target_subject.fs,
)
len(window_manager.filtered_r_peaks_windows)

Feature extraction


In [ ]:
templates = initialize_templates(window_manager.filtered_r_peaks_windows)
templates[0].shape

### Distribuições


In [ ]:
plot_age_distribution(df)

In [ ]:
plot_gender_distribution(df)

In [ ]:
normal_times, arrhythmia_times = split_annotations_by_type(annotation, fs)
normal_count = len(normal_times)
arrythm_count = len(arrhythmia_times)
print(normal_count, arrythm_count)
plot_annotation_counts(normal_count, arrythm_count)

In [ ]:
normal_count / (normal_count + arrythm_count) * 100

### Feature extraction


In [ ]:
extract_average_energy(window_manager.filtered_r_peaks_windows[0])

### Subamostragem
